# Progress 4 — Zero-shot LLM Baseline (Colab / LM Studio)

Notebook ini menyiapkan run zero-shot IdSarcasm. Jalur utama adalah HuggingFace log-probability scoring (`hf-logprobs`) karena paling dekat dengan paper. Jalur alternatif adalah LM Studio/OpenAI-compatible API untuk model lokal quantized.

**Saran:** mulai dari smoke test, lalu full Twitter, lalu full Reddit. Semua run menyimpan metrics, predictions, logs, dan waktu running.


## 1. Setup repo dan dependencies

Jika notebook dijalankan di Colab baru, jalankan cell ini. Kalau repo sudah ada, cell ini tidak akan clone ulang.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/feb027/idsarcasm-reproduction.git"
REPO_DIR = Path("/content/idsarcasm-reproduction")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only
!pip install -q -r requirements.txt


## 2. Cek GPU

Untuk full run HuggingFace, Colab GPU lebih disarankan.


In [ ]:
import torch, platform
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Smoke test wajib

Smoke test memakai `--max-samples 8`, jadi output masuk ke `results/tables/zeroshot_smoke.csv`, bukan tabel final.


In [ ]:
!python scripts/run_zeroshot_baseline.py --dataset twitter --model mt0-small --backend hf-logprobs --max-samples 8 --dtype float16 --device-map auto --disable-tqdm --write-log


## 4. Cek hasil smoke test


In [ ]:
!cat results/tables/zeroshot_smoke.csv
!find results/zeroshot -maxdepth 2 -type f | sort | head -50
!ls -lh results/logs/progress-4-zeroshot-*.log


## 5. Full run minimal Progress 4

Minimal untuk laporan: satu model (`mt0-small`) pada Twitter dan Reddit. Runtime Reddit jauh lebih lama karena test set 2.824 data × 5 prompt.


In [ ]:
# Twitter full run
!python scripts/run_zeroshot_baseline.py --dataset twitter --model mt0-small --backend hf-logprobs --dtype float16 --device-map auto --disable-tqdm --write-log

# Reddit full run
!python scripts/run_zeroshot_baseline.py --dataset reddit --model mt0-small --backend hf-logprobs --dtype float16 --device-map auto --disable-tqdm --write-log


## 6. Optional: BLOOMZ-560M sebagai pembanding

Jalankan jika waktu Colab masih cukup. Ini memberi pembanding dari keluarga BLOOMZ dan mT0.


In [ ]:
# Optional Twitter + Reddit BLOOMZ-560M
# !python scripts/run_zeroshot_baseline.py --dataset twitter --model bloomz-560m --backend hf-logprobs --dtype float16 --device-map auto --disable-tqdm --write-log
# !python scripts/run_zeroshot_baseline.py --dataset reddit --model bloomz-560m --backend hf-logprobs --dtype float16 --device-map auto --disable-tqdm --write-log


## 7. Inspect hasil final dan runtime


In [ ]:
import pandas as pd
from pathlib import Path

path = Path("results/tables/zeroshot_baselines.csv")
if path.exists():
    df = pd.read_csv(path)
    display(df)
    display(df[["dataset", "backend", "model_alias", "f1", "runtime_seconds", "avg_latency_seconds", "num_examples", "prompt_count"]])
else:
    print("Belum ada results/tables/zeroshot_baselines.csv")


## 8. LM Studio / local OpenAI-compatible backend

Cell ini hanya dipakai jika notebook dijalankan di mesin lokal yang bisa mengakses LM Studio server. Di Colab, `localhost:1234` mengarah ke mesin Colab, bukan PC Anda.


In [ ]:
# Contoh untuk lokal, bukan Colab:
# !python scripts/run_zeroshot_baseline.py --dataset twitter --model local-model --backend openai-compatible --api-base http://localhost:1234/v1 --max-samples 20 --disable-tqdm --write-log


## 9. Scan log sebelum commit

Pastikan tidak ada token/API key di log.


In [ ]:
!grep -RniE "hf_[A-Za-z0-9]|api[_-]?key|password|secret|Bearer" results/logs/progress-4-zeroshot-*.log || true


## 10. Commit hasil setelah full run

Jalankan setelah hasil full sudah siap.


In [ ]:
# !git status --short
# !git add results/tables/zeroshot_baselines.csv results/tables/zeroshot_smoke.csv results/zeroshot/ results/logs/progress-4-zeroshot-*.log
# !git commit -m "results: add Progress 4 zero-shot baseline runs"
# !git push
